In [ ]:
import re
from collections import deque

def parse_line(line: str):
    pattern = re.search(r'\[([.#]+)\]', line).group(1)

    button_strs = re.findall(r'\(([^)]*)\)', line)

    buttons = []
    for s in button_strs:
        s = s.strip()
        if not s:
            continue
        idxs = [int(x) for x in s.split(',')]
        mask = 0
        for i in idxs:
            mask |= 1 << i
        buttons.append(mask)

    return pattern, buttons

def pattern_to_mask(pattern: str) -> int:
    mask = 0
    for i, ch in enumerate(pattern):
        if ch == '#':
            mask |= 1 << i
    return mask

def min_presses_for_machine(pattern: str, buttons: list[int]) -> int | None:
    target = pattern_to_mask(pattern)
    start = 0

    if target == start:
        return 0

    queue = deque([start])
    dist = {start: 0} 

    while queue:
        state = queue.popleft()
        d = dist[state]

        for bmask in buttons:
            new_state = state ^ bmask 

            if new_state not in dist:
                if new_state == target:
                    return d + 1
                dist[new_state] = d + 1
                queue.append(new_state)

    return None

t = open("input.txt").read()
# t = """[.##.] (3) (1,3) (2) (2,3) (0,2) (0,1) {3,5,4,7}
# [...#.] (0,2,3,4) (2,3) (0,4) (0,1,2) (1,2,3,4) {7,5,12,7,2}
# [.###.#] (0,1,2,3,4) (0,3,4) (0,1,2,4,5) (1,2) {10,11,11,5,10,5}
# """
total = 0
for line in t.strip().splitlines():
    line = line.strip()
    if not line:
        continue

    pattern, buttons = parse_line(line)
    presses = min_presses_for_machine(pattern, buttons)
    
    total += presses
print(total)


500


In [3]:
import re
from ortools.sat.python import cp_model


def parse_line_joltage(line: str):

    pattern_match = re.search(r'\[([.#]+)\]', line)
    pattern = pattern_match.group(1) if pattern_match else None

    button_strs = re.findall(r'\(([^)]*)\)', line)
    buttons = []
    for s in button_strs:
        s = s.strip()
        if not s:
            continue
        idxs = [int(x) for x in s.split(',')]
        buttons.append(idxs)

    jolts_match = re.search(r'\{([^}]*)\}', line)
    if jolts_match:
        jolts = [int(x) for x in jolts_match.group(1).split(',')]
    else:
        jolts = []

    return pattern, buttons, jolts


def min_presses_joltage_ortools(target, buttons):

    n_counters = len(target)
    n_buttons = len(buttons)

    if n_counters == 0:
        return 0

    counters_touched_by = [[] for _ in range(n_counters)]
    for j, idxs in enumerate(buttons):
        for i in idxs:
            if 0 <= i < n_counters:
                counters_touched_by[i].append(j)

    for i, t in enumerate(target):
        if t > 0 and not counters_touched_by[i]:
            return None

    button_upper = []
    for j, idxs in enumerate(buttons):
        valid_idxs = [i for i in idxs if 0 <= i < n_counters]
        if not valid_idxs:
            button_upper.append(0)
        else:
            button_upper.append(min(target[i] for i in valid_idxs))

    model = cp_model.CpModel()

    x = []
    for j in range(n_buttons):
        ub = max(0, button_upper[j])
        var = model.NewIntVar(0, ub, f"x_{j}")
        x.append(var)

    model.Minimize(sum(x))

    for i in range(n_counters):
        model.Add(sum(x[j] for j in counters_touched_by[i]) == target[i])

    solver = cp_model.CpSolver()
    solver.log_search_progress = False
    solver.parameters.log_search_progress = False

    status = solver.Solve(model)

    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        return None

    total_presses = sum(int(solver.Value(var)) for var in x)
    return total_presses


# t = """[.##.] (3) (1,3) (2) (2,3) (0,2) (0,1) {3,5,4,7}
# [...#.] (0,2,3,4) (2,3) (0,4) (0,1,2) (1,2,3,4) {7,5,12,7,2}
# [.###.#] (0,1,2,3,4) (0,3,4) (0,1,2,4,5) (1,2) {10,11,11,5,10,5}
# """
t = open("input.txt").read()

total=0
for line in t.strip().splitlines():
    line = line.strip()
    if not line:
        continue

    _, buttons, target = parse_line_joltage(line)
    presses = min_presses_joltage_ortools(target, buttons)
    total += presses
print(total)




19763
